# Execution — Multi-Dataset Benchmark

Runs the four ablation configurations (no-learning, online-only, offline-only, combined) on each benchmark family, producing four CSVs per dataset (20 CSV total). Fixed settings come from `tuning_results_backup/best_config_evaluation.json` (Idris's Optuna run); the GBT repair model and the pre-trained LinUCB bandit state are loaded once and shared by all configurations that use them.

In [ ]:
import json
import warnings
from importlib.resources import files

from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
    hybrid_alns_solver,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.models import (
    load_repair_model,
    load_bandit_state,
)
from bin_packing_optimization.utilities.benchmarking import create_benchmark
import bin_packing_optimization.utilities.graphing as graphing
import bin_packing_optimization.utilities.statistics as statistics

warnings.filterwarnings("ignore", message="method=.*is ignored")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

## Setup — change `REPAIR_MODEL_FILE` and `BANDIT_STATE_FILE` when new artifacts arrive

In [ ]:
# ===========================================================================
# === The ONLY two lines to edit when Adem / the bandit training delivers ===
# ===========================================================================
REPAIR_MODEL_FILE = "repair_model_v3.pkl"     # ← swap to repair_model_v3.pkl when Adem ships
BANDIT_STATE_FILE = "bandit_state_v1.pkl"     # ← pre-trained bandit (None to cold-start)
# ===========================================================================

# Load tuned hyperparameters from the Optuna tuning run.
# Prefer the fresh `tuning_results/` directory (written by tune_with_optuna.py);
# fall back to the committed `tuning_results_backup/` if no fresh run is around.
_tuning_root = files(
    "bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.parameter_tuning"
)
TUNED_PARAMS = None
for _candidate in ("tuning_results", "tuning_results_backup"):
    _path = _tuning_root / _candidate / "best_config_evaluation.json"
    if _path.is_file():
        with _path.open(encoding="utf-8") as _f:
            TUNED_PARAMS = json.load(_f)["parameters"]
        print(f"Loaded tuned params from {_candidate}/")
        break
if TUNED_PARAMS is None:
    raise FileNotFoundError(
        "best_config_evaluation.json not found in tuning_results/ or tuning_results_backup/. "
        "Run tune_with_optuna.py first to generate it."
    )

# Load shared artifacts once (re-used by every configuration that needs them)
_repair_model = load_repair_model(REPAIR_MODEL_FILE)
try:
    _bandit_state = load_bandit_state(BANDIT_STATE_FILE) if BANDIT_STATE_FILE else None
except FileNotFoundError:
    print(f"[!] {BANDIT_STATE_FILE!r} not found, falling back to cold-start bandit "
          f"(run bandit_training.py to generate it).")
    _bandit_state = None

# Base hyperparameters for every configuration
BASE_ARGS = {"max_iterations": 5000, **TUNED_PARAMS}


def _config(use_offline: bool, use_online: bool) -> dict:
    args = dict(BASE_ARGS)
    args["use_offline_model"] = use_offline
    args["use_online_rl"] = use_online
    if use_offline:
        args["model_bundle"] = _repair_model
    if use_online and _bandit_state is not None:
        args["bandit_state"] = _bandit_state
    return args


CONFIGS = {
    "no-learning":  _config(use_offline=False, use_online=False),
    "online-only":  _config(use_offline=False, use_online=True),
    "offline-only": _config(use_offline=True,  use_online=False),
    "combined":     _config(use_offline=True,  use_online=True),
}


def run_all_configs(dataset_key: str, csv_name: str, time_limit: float, max_instances: int) -> dict:
    """Run all four ablation configurations on a dataset; return {config: csv_path}."""
    csvs = {}
    for cfg_name, cfg_args in CONFIGS.items():
        b = create_benchmark(dataset_key, hybrid_alns_solver, time_limit=time_limit)
        b.run(
            method=cfg_name,
            method_args=cfg_args,
            max_instances=max_instances,
            workers=8,
        )
        csvs[cfg_name] = b.save_results_to_csv(
            f"results/performance_evaluation/multi_dataset/{csv_name}_{cfg_name}.csv"
        )
    return csvs


print(f"Tuned params: {list(TUNED_PARAMS.keys())}")
print(f"Repair model: {REPAIR_MODEL_FILE}")
print(f"Bandit state: {BANDIT_STATE_FILE}"
      f"  (calls={_bandit_state.get('calls') if _bandit_state else 'n/a'})")

## Scholl-2

In [ ]:
scholl_2_csvs = run_all_configs(
    dataset_key='scholl-2',
    csv_name='scholl-2',
    time_limit=50.0,
    max_instances=830,
)

In [ ]:
# Print + graph the combined run as the headline; other configs are on disk.
statistics.print_benchmark_report(scholl_2_csvs["combined"])
graphing.display_graphs(scholl_2_csvs["combined"])

## Falkenauer-T

In [ ]:
falkenauer_t_csvs = run_all_configs(
    dataset_key='falkenauer-t',
    csv_name='falkenauer-t',
    time_limit=85.0,
    max_instances=508,
)

In [ ]:
# Print + graph the combined run as the headline; other configs are on disk.
statistics.print_benchmark_report(falkenauer_t_csvs["combined"])
graphing.display_graphs(falkenauer_t_csvs["combined"])

## Falkenauer-U

In [ ]:
falkenauer_u_csvs = run_all_configs(
    dataset_key='falkenauer-u',
    csv_name='falkenauer-u',
    time_limit=200.0,
    max_instances=141,
)

In [ ]:
# Print + graph the combined run as the headline; other configs are on disk.
statistics.print_benchmark_report(falkenauer_u_csvs["combined"])
graphing.display_graphs(falkenauer_u_csvs["combined"])

## Wäscher

In [ ]:
wascher_csvs = run_all_configs(
    dataset_key='wäscher',
    csv_name='wascher',
    time_limit=100.0,
    max_instances=425,
)

In [ ]:
# Print + graph the combined run as the headline; other configs are on disk.
statistics.print_benchmark_report(wascher_csvs["combined"])
graphing.display_graphs(wascher_csvs["combined"])

## Hard28

In [ ]:
hard28_csvs = run_all_configs(
    dataset_key='hard28',
    csv_name='hard28',
    time_limit=500.0,
    max_instances=50,
)

In [ ]:
# Print + graph the combined run as the headline; other configs are on disk.
statistics.print_benchmark_report(hard28_csvs["combined"])
graphing.display_graphs(hard28_csvs["combined"])

## Cross-dataset aggregate (combined config across families)

In [ ]:
_combined_csvs = [
        scholl_2_csvs["combined"],
        falkenauer_t_csvs["combined"],
        falkenauer_u_csvs["combined"],
        wascher_csvs["combined"],
        hard28_csvs["combined"],
    ]
statistics.print_multi_benchmark_report(_combined_csvs)

In [ ]:
graphing.display_multi_dataset_graphs(
    _combined_csvs,
    out_dir="results/performance_evaluation/multi_dataset/graphs",
)